# Notebook 1: Trust Inference Demo
**Paper:** *Agents for Agents: An Interrogator-Based Secure Framework for Autonomous IoUT*

This notebook demonstrates:
1. Generating synthetic behavioral sequences
2. Training the lightweight transformer trust model
3. Running inference and computing trust scores
4. Visualising the score distributions

---
> **Note:** All data is synthetic. Paper under review.

In [ ]:
# ── Cell 1: Install dependencies (Colab) ──────────────────────────────────
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'torch', 'numpy', 'pandas', 'matplotlib', 'tqdm'], check=True)
print('Dependencies ready.')

In [ ]:
# ── Cell 2: Clone repository (Colab only) ─────────────────────────────────
import os
if not os.path.exists('IoUT-Interrogator-Framework'):
    subprocess.run(['git', 'clone',
                    'https://github.com/aliakarma/IoUT-Interrogator-Framework.git'],
                   check=True)
os.chdir('IoUT-Interrogator-Framework')
sys.path.insert(0, '.')
print('Working directory:', os.getcwd())

In [ ]:
# ── Cell 3: Imports ───────────────────────────────────────────────────────
import json
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader

from simulation.scripts.generate_behavioral_data import generate_dataset
from model.inference.transformer_model import (
    TrustTransformer, TrustDataset, train_epoch, evaluate
)

torch.manual_seed(42)
np.random.seed(42)
print('Imports OK.')

In [ ]:
# ── Cell 4: Generate synthetic behavioral sequences (v3 — harder data) ─
sequences, label_df = generate_dataset(
    num_sequences=300, K=64, adv_fraction=0.15, seed=42
)

print(f'Total sequences: {len(sequences)}')
print(f'Adversarial:     {label_df["label"].sum()} ({label_df["label"].mean()*100:.1f}%)')
print(f'Legitimate:      {(1-label_df["label"]).sum()} ({(1-label_df["label"]).mean()*100:.1f}%)')
print('\nAttack breakdown (includes low_and_slow mimicry):')
print(label_df[label_df.label==1].attack_type.value_counts())

# Verify AR(1) temporal correlation
import numpy as np
legit_sample = [s for s in sequences if s['label']==0][0]
arr = np.array(legit_sample['sequence'])
autocorrs = [np.corrcoef(arr[:-1,i], arr[1:,i])[0,1] for i in range(5)]
print(f'\nAR(1) autocorrelations per feature: {[f"{a:.2f}" for a in autocorrs]}')
print('Positive values confirm temporal correlation (v3 fix)')


In [ ]:
# ── Cell 5: Visualise example sequences ───────────────────────────────────
feature_names = [
    'Timing Variance', 'Retx Frequency',
    'Routing Stability', 'Neighbor Churn', 'Protocol Compliance'
]

legit_seq = next(s for s in sequences if s['label'] == 0)
adv_seq   = next(s for s in sequences if s['label'] == 1)

fig, axes = plt.subplots(2, 1, figsize=(12, 6))
for ax, seq_data, title, color in [
    (axes[0], legit_seq, 'Legitimate Agent Behavioral Sequence', '#2171b5'),
    (axes[1], adv_seq,   f'Adversarial Agent ({adv_seq["attack_type"]})', '#d7191c'),
]:
    seq = np.array(seq_data['sequence'])
    for i, fname in enumerate(feature_names):
        ax.plot(seq[:, i], label=fname, alpha=0.8, linewidth=1.2)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_xlabel('Monitoring Window')
    ax.set_ylabel('Feature Value (0-1)')
    ax.set_ylim(0, 1)
    ax.legend(loc='upper right', fontsize=8, ncol=3)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('analysis/plots/example_sequences.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved: analysis/plots/example_sequences.png')

In [ ]:
# ── Cell 6: Build the transformer model ───────────────────────────────────
with open('model/configs/transformer_config.json') as f:
    config = json.load(f)

model = TrustTransformer(config)
print(f'Model parameters: {model.count_parameters():,}')
print(f'Architecture: {config["architecture"]["num_layers"]} layers, '
      f'd_model={config["architecture"]["d_model"]}, '
      f'heads={config["architecture"]["num_heads"]}')

In [ ]:
# ── Cell 7: Quick training (10 epochs for demo) ───────────────────────────
import random
random.shuffle(sequences)
n = len(sequences)
n_train = int(n * 0.7)
n_val   = int(n * 0.15)
train_seqs = sequences[:n_train]
val_seqs   = sequences[n_train:n_train + n_val]
test_seqs  = sequences[n_train + n_val:]

seq_len = config['architecture']['seq_len']
train_loader = DataLoader(TrustDataset(train_seqs, seq_len), batch_size=32, shuffle=True)
val_loader   = DataLoader(TrustDataset(val_seqs, seq_len),   batch_size=32)
test_loader  = DataLoader(TrustDataset(test_seqs, seq_len),  batch_size=32)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = torch.nn.BCELoss()

train_accs, val_accs = [], []
for epoch in range(1, 11):   # 10 epochs for Colab demo
    t_loss, t_acc = train_epoch(model, train_loader, optimizer, criterion)
    v_loss, v_acc, v_prec, v_rec = evaluate(model, val_loader, criterion)
    train_accs.append(t_acc)
    val_accs.append(v_acc)
    print(f'Epoch {epoch:2d} | Train Acc: {t_acc:.4f} | Val Acc: {v_acc:.4f} '
          f'Prec: {v_prec:.4f} Rec: {v_rec:.4f}')

In [ ]:
# ── Cell 8: Training curve ─────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 4))
epochs = range(1, len(train_accs) + 1)
ax.plot(epochs, [a * 100 for a in train_accs], label='Train Accuracy', color='#2171b5')
ax.plot(epochs, [a * 100 for a in val_accs],   label='Val Accuracy',   color='#2ca02c')
ax.axhline(y=94.2, color='gray', linestyle='--', alpha=0.6, label='Paper reported (94.2%)')
ax.set_xlabel('Epoch')
ax.set_ylabel('Accuracy (%)')
ax.set_title('Trust Transformer Training Curve (10-epoch demo)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('analysis/plots/training_curve.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# ── Cell 9: Trust score distribution ──────────────────────────────────────
model.eval()
scores, labels = [], []
with torch.no_grad():
    for seq_tensor, lbl_tensor in DataLoader(TrustDataset(test_seqs, seq_len), batch_size=64):
        out = model(seq_tensor).squeeze().numpy()
        scores.extend(out.tolist())
        labels.extend(lbl_tensor.squeeze().numpy().tolist())

scores = np.array(scores)
labels = np.array(labels)

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(scores[labels == 0], bins=30, alpha=0.7, color='#2171b5', label='Legitimate agents')
ax.hist(scores[labels == 1], bins=30, alpha=0.7, color='#d7191c', label='Adversarial agents')
ax.axvline(x=0.65, color='black', linestyle='--', linewidth=2, label='τ_min = 0.65')
ax.set_xlabel('Trust Score τ')
ax.set_ylabel('Count')
ax.set_title('Trust Score Distribution — Legitimate vs Adversarial Agents')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('analysis/plots/trust_score_distribution.png', dpi=120, bbox_inches='tight')
plt.show()

# Accuracy
preds = (scores < 0.65).astype(int)
acc = (preds == labels).mean()
print(f'Test accuracy: {acc:.4f}')